In [1]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rc
from sklearn.feature_extraction.text import CountVectorizer
from konlpy.tag import Mecab
from scipy import sparse
from kneed import KneeLocator
from tqdm import tqdm

# ---------------------------------------------------------
# 1. 설정 및 폴더 생성
# ---------------------------------------------------------
# [수정됨] 분석할 대상 폴더들을 리스트로 관리
TARGET_DIRS = [
    "data/대법원_판례_txt화", 
    "data/하위법원_판례_txt화"
]

OUTPUT_DIR = "stopword_판례_통합"  # 결과 저장할 폴더 이름도 변경 (선택사항)

# 폴더가 없으면 생성
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f">>> 결과 저장 폴더 생성 완료: {OUTPUT_DIR}")

# Mac 한글 폰트 설정 (Windows라면 'Malgun Gothic' 등으로 변경 필요)
rc('font', family='AppleGothic')
plt.rcParams['axes.unicode_minus'] = False

# ---------------------------------------------------------
# 2. 데이터 로드 (통합 버전)
# ---------------------------------------------------------
mecab = Mecab()

def get_corpus(target_dirs_list):
    all_files = []
    
    # [수정됨] 여러 폴더를 순회하며 파일 리스트 취합
    print(">>> 파일 리스트 수집 중...")
    for target_dir in target_dirs_list:
        files = glob.glob(os.path.join(target_dir, "**", "*.txt"), recursive=True)
        all_files.extend(files)
        print(f"   - '{target_dir}' 폴더: {len(files)}개 파일 발견")
    
    total_files = len(all_files)
    print(f">>> 총 {total_files}개 파일에 대해 전처리 시작...")
    
    corpus = []
    # 전체 파일에 대해 진행률 표시 (tqdm)
    for file_path in tqdm(all_files, desc="형태소 분석 진행"):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read()
            # 명사(N), 용언(V)만 추출
            tokens = [w for w, p in mecab.pos(content) if p.startswith('N') or p.startswith('V')]
            if tokens: 
                corpus.append(" ".join(tokens))
        except Exception as e:
            # 읽기 오류 발생 시 경고 출력 후 건너뜀
            # print(f"Error reading {file_path}: {e}") 
            continue
            
    return corpus

# 함수 호출 시 리스트 전달
corpus_data = get_corpus(TARGET_DIRS)

if not corpus_data:
    print("[Error] 데이터를 찾지 못했습니다. 경로를 확인해주세요.")
    exit()

print(f">>> 전처리 완료된 문서 수: {len(corpus_data)}건")

# ---------------------------------------------------------
# 3. 지표 계산 (DTM -> Metrics)
# ---------------------------------------------------------
print(">>> 3가지 기법(Zipf, Entropy, MP-VP) 지표 계산 중...")

# DTM 생성
vectorizer = CountVectorizer(min_df=5) 
X = vectorizer.fit_transform(corpus_data)
features = np.array(vectorizer.get_feature_names_out())

# (1) TF (Zipf)
count = np.array(X.sum(axis=0)).flatten()

# (2) Entropy
total_freq = count
inv_freq = 1.0 / total_freq
D_inv = sparse.diags(inv_freq)
P_dt = X @ D_inv  
P_dt.data = P_dt.data * np.log(P_dt.data)
entropy = -1 * np.array(P_dt.sum(axis=0)).flatten()

# (3) MP-VP
doc_len = np.array(X.sum(axis=1)).flatten()
doc_len[doc_len == 0] = 1
D_inv_doc = sparse.diags(1.0 / doc_len)
P_td = D_inv_doc @ X  

mp = np.array(P_td.mean(axis=0)).flatten()
P_td_sq = P_td.copy()
P_td_sq.data **= 2
vp = np.array(P_td_sq.mean(axis=0)).flatten() - (mp**2)

# ---------------------------------------------------------
# 4. 데이터프레임 통합 및 정규화
# ---------------------------------------------------------
df = pd.DataFrame({
    'word': features, 
    'count': count, 
    'entropy': entropy, 
    'mp': mp, 
    'vp': vp
})

# 정규화 (Normalization)
for col in ['count', 'entropy', 'mp', 'vp']:
    df[f'norm_{col}'] = (df[col] - df[col].min()) / (df[col].max() - df[col].min())

# 최종 점수 산출
df['score_zipf'] = df['norm_count']
df['score_entropy'] = df['norm_entropy']
df['score_mpvp'] = df['norm_mp'] + (1 - df['norm_vp'])

# 전체 분석 결과 저장 (소수점 4자리)
df.round(4).to_csv(os.path.join(OUTPUT_DIR, "statute_analysis_full.csv"), index=False, encoding='utf-8-sig')
print(f">>> 전체 분석 데이터 저장 완료: {OUTPUT_DIR}/statute_analysis_full.csv")

# ---------------------------------------------------------
# 5. 엘보우 분석, 시각화 및 개별 리스트 저장
# ---------------------------------------------------------
def process_elbow_method(df, score_col, method_name, filename_suffix):
    # 정렬
    sorted_df = df.sort_values(by=score_col, ascending=False).reset_index(drop=True)
    
    x = sorted_df.index.tolist()
    y = sorted_df[score_col].tolist()
    
    # Kneedle
    kneedle = KneeLocator(x, y, curve="convex", direction="decreasing", S=1.0)
    elbow_rank = kneedle.elbow
    
    if elbow_rank is None:
        elbow_rank = 50 # fallback
        
    elbow_value = y[elbow_rank]
    elbow_word = sorted_df.iloc[elbow_rank]['word']
    
    # 후보군 추출
    candidates = sorted_df.head(elbow_rank)
    
    # 그래프 그리기 & 저장
    plt.figure(figsize=(10, 6))
    plt.plot(x, y, label=f'{method_name} Score')
    plt.vlines(elbow_rank, plt.ylim()[0], plt.ylim()[1], linestyles='dashed', colors='red', label='Elbow Point')
    plt.scatter([elbow_rank], [elbow_value], color='red', s=100, zorder=5)
    
    plt.title(f"[통합 판례] {method_name} Score Distribution")
    plt.xlabel("Rank")
    plt.ylabel("Score")
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.annotate(f'Cut-off: {elbow_rank}위\nWord: {elbow_word}', 
                 xy=(elbow_rank, elbow_value), 
                 xytext=(elbow_rank + 50, elbow_value + 0.05),
                 arrowprops=dict(facecolor='black', shrink=0.05))
    
    # 이미지 파일 저장
    img_path = os.path.join(OUTPUT_DIR, f"graph_{filename_suffix}.png")
    plt.savefig(img_path)
    plt.close() # 메모리 해제
    
    # 후보 리스트 CSV 저장
    csv_path = os.path.join(OUTPUT_DIR, f"candidates_{filename_suffix}.csv")
    candidates.to_csv(csv_path, index=False, encoding='utf-8-sig')
    
    print(f"[{method_name}] Elbow: {elbow_rank}위 ('{elbow_word}') -> 저장됨")
    
    return set(candidates['word'])

print("\n>>> 엘보우 분석 및 시각화 진행 중...")

# A. Zipf
set_a = process_elbow_method(df, 'score_zipf', 'Zipf(Frequency)', 'A_zipf')
# B. Entropy
set_b = process_elbow_method(df, 'score_entropy', 'Entropy', 'B_entropy')
# C. MP-VP
set_c = process_elbow_method(df, 'score_mpvp', 'Stability(MP-VP)', 'C_mpvp')

# ---------------------------------------------------------
# 6. 교집합(E) 및 합집합(D) 저장
# ---------------------------------------------------------
# 교집합 (Golden List for Statutes)
set_e = set_a.intersection(set_b).intersection(set_c)
pd.DataFrame({'word': list(set_e)}).to_csv(os.path.join(OUTPUT_DIR, "candidates_E_intersection.csv"), index=False, encoding='utf-8-sig')

# 합집합
set_d = set_a.union(set_b).union(set_c)
pd.DataFrame({'word': list(set_d)}).to_csv(os.path.join(OUTPUT_DIR, "candidates_D_union.csv"), index=False, encoding='utf-8-sig')

print("\n" + "="*60)
print(f"🏛️ 통합 판례 불용어 분석 완료! (결과 폴더: ./{OUTPUT_DIR})")
print(f"   - 교집합(Exp E) 개수: {len(set_e)}개")
if len(set_e) > 0:
    print(f"   - 주요 단어: {list(set_e)[:10]}")
print("="*60)

>>> 결과 저장 폴더 생성 완료: stopword_판례_통합
>>> 파일 리스트 수집 중...
   - 'data/대법원_판례_txt화' 폴더: 5501개 파일 발견
   - 'data/하위법원_판례_txt화' 폴더: 27729개 파일 발견
>>> 총 33230개 파일에 대해 전처리 시작...


형태소 분석 진행: 100%|██████████| 33230/33230 [02:59<00:00, 184.78it/s]


>>> 전처리 완료된 문서 수: 33228건
>>> 3가지 기법(Zipf, Entropy, MP-VP) 지표 계산 중...
>>> 전체 분석 데이터 저장 완료: stopword_판례_통합/statute_analysis_full.csv

>>> 엘보우 분석 및 시각화 진행 중...
[Zipf(Frequency)] Elbow: 187위 ('산정') -> 저장됨
[Entropy] Elbow: 519위 ('주위') -> 저장됨
[Stability(MP-VP)] Elbow: 221위 ('통하') -> 저장됨

🏛️ 통합 판례 불용어 분석 완료! (결과 폴더: ./stopword_판례_통합)
   - 교집합(Exp E) 개수: 141개
   - 주요 단어: ['관계', '의한', '변호사', '관리', '지급', '검사', '제공', '사이', '발생', '법리']
